# Fatias da imagem, círculos e extensão da aorta

Analisa a cobertura axial dos círculos da aorta e compara essa trajetória com as fatias efetivamente ocupadas pela máscara final. Valores positivos na variação indicam expansão axial da segmentação; valores negativos indicam recuo.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Localiza a raiz antes dos imports internos quando o notebook abre em src/eda.
current = Path.cwd().resolve()
REPO_ROOT = next(
    path for path in [current, *current.parents]
    if (path / "src").exists() and (path / "output").exists()
)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.experiments import (  # noqa: E402
    get_aorta_visual_review,
    load_aorta_review_cohort,
    load_aorta_visual_reviews,
)
from utils.project.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

## 1. Configuração

As duas coortes usam o pipeline normal com limite inferior de -300 HU e limite superior P99.9. Os rótulos de aorta boa/ruim vêm da inspeção visual dos HTMLs 3D.

In [2]:
REVIEW_CONFIG_PATH = REPO_ROOT / "config/aorta_visual_reviews.json"
REVIEW_CATALOG = load_aorta_visual_reviews(REVIEW_CONFIG_PATH)
TRAIN_REVIEW = get_aorta_visual_review(REVIEW_CATALOG, "normal", "train")
VAL_REVIEW = get_aorta_visual_review(REVIEW_CATALOG, "normal", "val")

print(
    f"Treino: {len(TRAIN_REVIEW['aorta_good_ids'])} boas e "
    f"{len(TRAIN_REVIEW['aorta_bad_ids'])} ruins"
)
print(
    f"Validação: {len(VAL_REVIEW['aorta_good_ids'])} boas e "
    f"{len(VAL_REVIEW['aorta_bad_ids'])} ruins"
)

Treino: 23 boas e 7 ruins
Validação: 47 boas e 13 ruins


## 2. Carregamento e métricas derivadas

`aorta_circle_count` mede as fatias representadas pela trajetória final de círculos. `aorta_segmented_slice_count` conta as fatias que possuem ao menos um voxel na máscara final da aorta.

In [3]:
REQUIRED_COLUMNS = {
    "IMG_ID",
    "image_slice_count",
    "aorta_circle_count",
    "aorta_detected_circle_count",
    "aorta_interpolated_circle_count",
    "aorta_circle_first_slice",
    "aorta_circle_last_slice",
    "aorta_circle_radius_mean_mm",
    "aorta_circle_radius_std_mm",
    "aorta_segmented_slice_count",
    "ostia_detection_status",
    "artery_dice",
}

train_df = load_aorta_review_cohort(
    REPO_ROOT,
    TRAIN_REVIEW,
    "train",
    cohort_name="treino",
    required_columns=REQUIRED_COLUMNS,
    use_reviewed_ostia_labels=True,
)
val_df = load_aorta_review_cohort(
    REPO_ROOT,
    VAL_REVIEW,
    "val",
    cohort_name="validação",
    required_columns=REQUIRED_COLUMNS,
    use_reviewed_ostia_labels=True,
)
combined_df = pd.concat([train_df, val_df], ignore_index=True)

print(f"Treino carregado: {len(train_df)} imagens")
print(f"Validação carregada: {len(val_df)} imagens")

Treino carregado: 30 imagens
Validação carregada: 60 imagens


## 3. Visão geral das coortes

In [4]:
def summarize_cohort(cohort_df, cohort_name):
    # Resume círculos, máscara final, raio e desempenho da coorte.
    return {
        "coorte": cohort_name,
        "imagens": len(cohort_df),
        "fatias_imagem_media": cohort_df["image_slice_count"].mean(),
        "fatias_com_circulo_media": cohort_df["aorta_circle_count"].mean(),
        "fatias_segmentadas_media": cohort_df["aorta_segmented_slice_count"].mean(),
        "variacao_fatias_media": cohort_df["segmented_minus_circle_slices"].mean(),
        "variacao_percentual_media": (
            cohort_df["segmented_vs_circle_change_fraction"].mean() * 100
        ),
        "cobertura_circulos_media_percentual": (
            cohort_df["circle_slice_fraction"].mean() * 100
        ),
        "cobertura_mascara_media_percentual": (
            cohort_df["segmented_slice_fraction"].mean() * 100
        ),
        "raio_medio_mm": cohort_df["aorta_circle_radius_mean_mm"].mean(),
        "sucesso_ostios_percentual": cohort_df["ostia_success"].mean() * 100,
        "dice_arterial_medio": cohort_df["artery_dice"].mean(),
    }


overview_df = pd.DataFrame(
    [
        summarize_cohort(train_df, "treino"),
        summarize_cohort(val_df, "validação"),
    ]
)
display(overview_df.round(3))

,coorte,imagens,fatias_imagem_media,fatias_com_circulo_media,fatias_segmentadas_media,variacao_fatias_media,variacao_percentual_media,cobertura_circulos_media_percentual,cobertura_mascara_media_percentual,raio_medio_mm,sucesso_ostios_percentual,dice_arterial_medio
0,treino,30,256.833,108.700,118.467,9.767,9.676,42.059,45.917,15.269,90.0,0.615
1,validação,60,256.633,105.617,115.417,9.800,10.186,40.982,44.833,15.366,80.0,0.565


## 4. Relação entre círculos e máscara final

A variação é calculada por

\[
\Delta N = N_{\mathrm{segmentadas}} - N_{\mathrm{círculos}}.
\]

- `ΔN > 0`: a máscara avançou além da quantidade de fatias representadas pelos círculos;
- `ΔN = 0`: as extensões são iguais;
- `ΔN < 0`: houve recuo axial após level set e pós-processamento.

A variação percentual usa a quantidade de fatias com círculos como referência.

In [5]:
extent_summary_df = (
    combined_df.assign(
        direcao_extensao=combined_df["segmented_minus_circle_slices"].map(
            lambda value: "expansão" if value > 0 else ("recuo" if value < 0 else "igual")
        )
    )
    .groupby(["coorte", "visual_aorta_quality"], observed=True)
    .agg(
        imagens=("IMG_ID", "size"),
        fatias_com_circulo_media=("aorta_circle_count", "mean"),
        fatias_segmentadas_media=("aorta_segmented_slice_count", "mean"),
        variacao_fatias_media=("segmented_minus_circle_slices", "mean"),
        variacao_fatias_mediana=("segmented_minus_circle_slices", "median"),
        variacao_percentual_media=("segmented_vs_circle_change_fraction", "mean"),
        exames_com_expansao=("direcao_extensao", lambda values: values.eq("expansão").sum()),
        exames_com_recuo=("direcao_extensao", lambda values: values.eq("recuo").sum()),
        exames_sem_variacao=("direcao_extensao", lambda values: values.eq("igual").sum()),
        sucesso_ostios_percentual=("ostia_success", "mean"),
    )
    .reset_index()
)
extent_summary_df["variacao_percentual_media"] *= 100
extent_summary_df["sucesso_ostios_percentual"] *= 100
display(extent_summary_df.round(3))

,coorte,visual_aorta_quality,imagens,fatias_com_circulo_media,fatias_segmentadas_media,variacao_fatias_media,variacao_fatias_mediana,variacao_percentual_media,exames_com_expansao,exames_com_recuo,exames_sem_variacao,sucesso_ostios_percentual
0,treino,boa,23,108.652,118.348,9.696,10.0,9.676,23,0,0,95.652
1,treino,ruim,7,108.857,118.857,10.000,10.0,9.675,7,0,0,71.429
2,validação,boa,47,101.404,111.149,9.745,10.0,10.234,47,0,0,91.489
3,validação,ruim,13,120.846,130.846,10.000,10.0,10.012,13,0,0,38.462


## 5. Círculos por qualidade visual da aorta

Esta tabela mantém as informações de cobertura, raio e posição que ajudam a distinguir trajetórias boas e ruins, sem repetir as mesmas relações em vários gráficos.

In [6]:
quality_summary_df = (
    combined_df.groupby(["coorte", "visual_aorta_quality"], observed=True)
    .agg(
        imagens=("IMG_ID", "size"),
        fatias_imagem_media=("image_slice_count", "mean"),
        cobertura_circulos_media=("circle_slice_fraction", "mean"),
        cobertura_mascara_media=("segmented_slice_fraction", "mean"),
        raio_medio_mm=("aorta_circle_radius_mean_mm", "mean"),
        variacao_media_raio_mm=("aorta_circle_radius_std_mm", "mean"),
        primeira_posicao_media=("circle_first_position", "mean"),
        centro_posicao_media=("circle_center_position", "mean"),
        ultima_posicao_media=("circle_last_position", "mean"),
        sucesso_ostios=("ostia_success", "mean"),
        dice_medio=("artery_dice", "mean"),
    )
    .reset_index()
)
percentage_columns = [
    "cobertura_circulos_media",
    "cobertura_mascara_media",
    "primeira_posicao_media",
    "centro_posicao_media",
    "ultima_posicao_media",
    "sucesso_ostios",
]
quality_summary_df[percentage_columns] *= 100
display(quality_summary_df.round(3))

,coorte,visual_aorta_quality,imagens,fatias_imagem_media,cobertura_circulos_media,cobertura_mascara_media,raio_medio_mm,variacao_media_raio_mm,primeira_posicao_media,centro_posicao_media,ultima_posicao_media,sucesso_ostios,dice_medio
0,treino,boa,23,262.217,41.035,44.770,15.656,1.463,58.965,79.290,99.615,95.652,0.622
1,treino,ruim,7,239.143,45.426,49.683,13.997,1.516,54.574,77.074,99.574,71.429,0.591
2,validação,boa,47,256.617,39.516,43.341,15.659,1.318,60.484,80.045,99.606,91.489,0.619
3,validação,ruim,13,256.692,46.283,50.226,14.306,1.281,53.717,76.661,99.606,38.462,0.369


## 6. Valores por exame

A tabela começa pelas aortas ruins e mostra diretamente quanto a máscara avançou ou recuou em relação à trajetória de círculos.

In [7]:
case_table_df = (
    combined_df.assign(
        cobertura_circulos_percentual=combined_df["circle_slice_fraction"] * 100,
        cobertura_mascara_percentual=combined_df["segmented_slice_fraction"] * 100,
        variacao_percentual=(
            combined_df["segmented_vs_circle_change_fraction"] * 100
        ),
    )
    .sort_values(
        ["coorte", "visual_aorta_quality", "segmented_minus_circle_slices"],
        ascending=[True, False, True],
    )
    [[
        "coorte",
        "IMG_ID",
        "visual_aorta_quality",
        "ostia_outcome",
        "image_slice_count",
        "aorta_circle_count",
        "aorta_segmented_slice_count",
        "segmented_minus_circle_slices",
        "variacao_percentual",
        "cobertura_circulos_percentual",
        "cobertura_mascara_percentual",
        "aorta_detected_circle_count",
        "aorta_interpolated_circle_count",
        "aorta_circle_radius_mean_mm",
        "aorta_circle_radius_std_mm",
        "artery_dice",
    ]]
)
display(case_table_df.round(3))

,coorte,IMG_ID,visual_aorta_quality,ostia_outcome,image_slice_count,aorta_circle_count,aorta_segmented_slice_count,segmented_minus_circle_slices,variacao_percentual,cobertura_circulos_percentual,cobertura_mascara_percentual,aorta_detected_circle_count,aorta_interpolated_circle_count,aorta_circle_radius_mean_mm,aorta_circle_radius_std_mm,artery_dice
2,treino,44,ruim,sucesso,206,94,104,10,10.638,45.631,50.485,94,0,12.901,1.416,0.616
5,treino,175,ruim,sucesso,231,105,115,10,9.524,45.455,49.784,105,0,14.429,1.883,0.458
9,treino,330,ruim,sucesso,275,153,163,10,6.536,55.636,59.273,153,0,14.040,1.734,0.681
17,treino,603,ruim,falha,206,68,78,10,14.706,33.010,37.864,68,0,13.303,1.093,0.449
18,treino,608,ruim,sucesso,206,114,124,10,8.772,55.340,60.194,114,0,14.697,1.538,0.668
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,validação,881,boa,sucesso,275,99,109,10,10.101,36.000,39.636,99,0,15.380,1.426,0.684
85,validação,886,boa,sucesso,275,136,146,10,7.353,49.455,53.091,136,0,15.045,1.647,0.618
87,validação,913,boa,sucesso,209,95,105,10,10.526,45.455,50.239,95,0,16.500,1.288,0.682
88,validação,961,boa,sucesso,206,79,89,10,12.658,38.350,43.204,79,0,15.259,1.290,0.696
